# Lab 6

Muhammad Usman | Roll No: 025

In [ ]:
!pip install -q scikit-learn sentence-transformers

## Task 1

In [ ]:
import re

samples = ["AI, search!", "I can't log in.", "help@example.com",
           "ABC123 version 2.1", "", "Dr. Smith said e.g. meet at 5 p.m."]
for t in samples:
    print(t)
    print("spaces:", t.split(), len(t.split()))
    print("words:", re.findall(r"\b\w+\b", t, flags=re.UNICODE))
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    for t in samples[:2]:
        print(t, model.tokenizer.tokenize(t))
except Exception as e:
    print("Model tokenizer not available here:", type(e).__name__)


## Task 2

In [ ]:
import re

def windows(text, size=40, overlap=10):
    spans = [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
    out, i = [], 0
    while i < len(spans):
        j = min(i + size, len(spans))
        out.append((text[spans[i][0]:spans[j - 1][1]], spans[i][0], spans[j - 1][1]))
        if j == len(spans):
            break
        i += size - overlap
    return out

def sentence_chunks(text, size=40):
    parts = [(m.start(), m.end()) for m in re.finditer(r"[^.!?]+[.!?]?", text) if m.group().strip()]
    out, start, end, count = [], None, None, 0
    for a, b in parts:
        n = len(text[a:b].split())
        if start is None:
            start, end, count = a, b, n
        elif count + n <= size:
            end, count = b, count + n
        else:
            out.append((text[start:end], start, end))
            start, end, count = a, b, n
    if start is not None:
        out.append((text[start:end], start, end))
    return out

if __name__ == "__main__":
    text = "Late work is penalized. Illness may qualify for an extension when medical evidence is provided. Ask the coordinator first."
    print("windows:", windows(text, 12, 4))
    print("sentences:", sentence_chunks(text, 12))


## Task 1 FAQ

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

faq = [
    ("How do I reset my password?", "Use the recovery link on the login page.", "it_help.txt"),
    ("Where do I submit my assignment?", "Use the assignment upload page in the learning portal.", "assessment.txt"),
    ("Can illness justify a late submission?", "Request an extension with medical evidence before the deadline.", "handbook.txt"),
    ("How do I check my ink level?", "Check the ink level on the printer screen.", "printer_help.txt"),
    ("Can I change my course after registration?", "Ask the academic office before the change deadline.", "registrar.txt"),
]
vec = TfidfVectorizer()
mat = vec.fit_transform([q + " " + a for q, a, s in faq])


def reply(query):
    if not query.strip():
        return "Please enter a question."
    for q, a, s in faq:
        if query.strip().casefold() == q.casefold():
            return a + " [source: " + s + "]"
    q = vec.transform([query])
    if q.nnz == 0:
        return "No matching FAQ. Please contact the help desk."
    scores = (q @ mat.T).toarray()[0]
    top = sorted(range(len(faq)), key=lambda i: -scores[i])[:3]
    return "Possible matches; choose or rephrase: " + ", ".join(faq[i][0] for i in top)


print("Exact:", reply("How do I reset my password?"))
print("Suggestion:", reply("password recovery"))
print("Empty:", reply(""))
print("Unsupported:", reply("volcano geology"))


## Data for Task 3

In [ ]:
import json
corpus = {'late': 'Late submission is normally penalized. Students who submit after the deadline lose marks for each day the work is late, and work submitted more than five days late is not marked at all. Illness extensions require medical evidence before the deadline. A student who was ill must send the medical certificate to the course coordinator before the deadline, not after the result is out. The coordinator checks the certificate and the dates on it, and then tells the student if extra days are allowed. A late request without medical evidence is treated as a normal late submission. The final file must still be uploaded as a PDF through the learning portal, even if an extension is granted. Students should keep a copy of the certificate and the email they sent, because the office may ask for it again. If the illness started after the deadline, the normal late rules apply and the extension rule does not help. This rule is about coursework only, and it does not change exam dates.', 'password': 'Reset a forgotten password using the recovery link. A student who cannot log in should click the recovery link on the login page and follow the steps shown there. The link is sent to the registered email address, and it only works for a short time. Never share recovery codes. Recovery codes are private, and no staff member will ever ask for them in an email or a message. If someone asks for a code, do not send it and report the message to the IT help desk. After the password is changed, log in again with the new password and check that old devices are signed out. If the recovery email does not arrive, check the spam folder first, then wait ten minutes before asking for a new link. Do not try the same link many times, because it can stop working. This guide is for student accounts only, and staff accounts follow a different process.', 'assignment': 'Upload assignments through the learning portal. Students must log in to the learning portal, open the right course, and use the assignment upload page for that task. Submit the final file as a PDF. The file name should include the student roll number and the assignment number, so the teacher can find it easily. A file uploaded in the wrong course or in the wrong format may not be marked. After upload, the portal shows a confirmation message, and students should take a screenshot of it. If the portal is slow near the deadline, upload early and do not wait until the last hour. Draft files and old versions should not be uploaded, because only the final file is marked. If the upload fails, try again once, then contact the help desk with the error message and the file name. Late upload rules are covered in the handbook, and they still apply even if the portal was slow.', 'contractor': "Contractors can work remotely only if the signed contract permits it. Employee rules do not apply to contractors. A contractor must read the contract first and check the remote work clause before working from home. If the contract says nothing about remote work, the contractor should ask the manager in writing and wait for a written reply. Working from home without permission can break the contract, even if employees in the same team work from home. The manager's approval for an employee does not cover a contractor. Contractors must also follow the same data rules when they work remotely, and they must not share work files on personal devices. If the contract is changed, the new version must be signed by both sides before the new rule starts. Questions about a contract should go to the HR office, not to other team members. A contractor who is unsure should not guess. The safest step is to send a short email to the manager and the HR office, keep the reply, and work only after the reply arrives. This protects both sides, because the written rule is then clear to everyone.", 'employee': 'Employees may request remote work with manager approval. An employee who wants to work from home should send a request to the manager and explain the days and the work they will do. The manager checks the team schedule and the type of work, and then gives a written decision. Remote work is not automatic, and a request can be refused if the work needs to be done on site. Approved employees must be online during work hours, attend meetings, and reply to messages in a reasonable time. If an employee works from home without approval, it is treated as absence. This rule is for regular employees only. It does not give contractors any right to work remotely, because contractors follow their signed contract instead. Managers should reply in writing, not only by speaking in a meeting, because a spoken reply is hard to prove later. The written reply should list the approved days, the work hours, and any condition that applies. If the team plan changes, the manager should tell the employee early, so the employee can plan the week.', 'printer': 'For blank printer pages, check ink levels and run the cleaning utility. Print a test page afterwards. A printer that has power but prints blank pages usually has an ink or head problem, not a power problem. First, check the ink level on the printer screen and replace any empty cartridge. Second, open the printer software and run the cleaning utility, then wait until it finishes. Third, print a test page and check if the text is clear. If the test page is still blank, run the cleaning utility one more time and check the paper type in the settings. Do not open the printer head by hand, because that can damage it. If the problem is still there after two cleanings, note the printer model and the error on the screen, and contact the IT help desk. This guide is for printers only, and it does not apply to scanners.'}
queries = [{'query': 'What is required for illness extensions?', 'doc': 'late', 'evidence': 'Illness extensions require medical evidence before the deadline.', 'split': 'dev'}, {'query': 'How do I reset a forgotten password?', 'doc': 'password', 'evidence': 'Reset a forgotten password using the recovery link.', 'split': 'dev'}, {'query': 'Where do I upload assignments?', 'doc': 'assignment', 'evidence': 'Upload assignments through the learning portal.', 'split': 'dev'}, {'query': 'Can contractors work remotely?', 'doc': 'contractor', 'evidence': 'Contractors can work remotely only if the signed contract permits it.', 'split': 'dev'}, {'query': 'What do I do for blank printer pages?', 'doc': 'printer', 'evidence': 'For blank printer pages, check ink levels and run the cleaning utility.', 'split': 'test'}, {'query': 'How can I recover my login access?', 'doc': 'password', 'evidence': 'Reset a forgotten password using the recovery link.', 'split': 'test'}, {'query': 'How should I hand in my coursework?', 'doc': 'assignment', 'evidence': 'Upload assignments through the learning portal.', 'split': 'test'}, {'query': 'What proof is needed when I was ill?', 'doc': 'late', 'evidence': 'Illness extensions require medical evidence before the deadline.', 'split': 'test'}, {'query': "What are tomorrow's flight prices?", 'doc': None, 'evidence': None, 'split': 'test'}, {'query': 'Who won the football match?', 'doc': None, 'evidence': None, 'split': 'test'}]
open('corpus.json','w').write(json.dumps(corpus))
open('queries.json','w').write(json.dumps(queries))
print('data written')

## Task 3

In [ ]:
import csv
from sklearn.feature_extraction.text import TfidfVectorizer

docs = corpus
queries = queries
ways = [("windows0", lambda t: windows(t, 40, 0)), ("windows10", lambda t: windows(t, 40, 10)), ("sentence", sentence_chunks)]
rows = []
for name, func in ways:
    chunks, owners, ids, starts, ends = [], [], [], [], []
    for doc_id, text in docs.items():
        for i, (c, a, b) in enumerate(func(text)):
            chunks.append(c)
            owners.append(doc_id)
            ids.append(doc_id + ":" + str(i))
            starts.append(a)
            ends.append(b)
    vec = TfidfVectorizer()
    mat = vec.fit_transform(chunks)
    for split in ["dev", "test"]:
        hits, rr, n = 0, 0, 0
        for q in queries:
            if q["split"] != split or q["doc"] is None:
                continue
            n += 1
            scores = (vec.transform([q["query"]]) @ mat.T).toarray()[0]
            top = sorted(range(len(chunks)), key=lambda i: -scores[i])[:3]
            for rank, i in enumerate(top, 1):
                if owners[i] == q["doc"] and q["evidence"] in chunks[i]:
                    hits += 1
                    rr += 1 / rank
                    break
        print(name, split, "Hit@3:", round(hits / n, 3), "MRR@3:", round(rr / n, 3))
    for q in queries:
        scores = (vec.transform([q["query"]]) @ mat.T).toarray()[0]
        top = sorted(range(len(chunks)), key=lambda i: -scores[i])[:3]
        for rank, i in enumerate(top, 1):
            print(name, q["query"], rank, round(float(scores[i]), 3), ids[i])
            rows.append({"strategy": name, "query": q["query"], "split": q["split"], "rank": rank,
                         "score": round(float(scores[i]), 3), "chunk_id": ids[i], "doc": owners[i],
                         "start": starts[i], "end": ends[i], "text": chunks[i]})
with open("retrieval_log.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)
print("Wrote retrieval_log.csv, rows:", len(rows))

## Task 3 Embedding

In [ ]:

docs = corpus
queries = queries
chunks, owners = [], []
for doc_id, text in docs.items():
    for c, a, b in sentence_chunks(text):
        chunks.append(c)
        owners.append(doc_id)
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
except Exception as e:
    print("Embedding not available here:", type(e).__name__)
    raise SystemExit
mat = model.encode(chunks, normalize_embeddings=True)
for split in ["dev", "test"]:
    hits, rr, n = 0, 0, 0
    for q in queries:
        if q["split"] != split or q["doc"] is None:
            continue
        n += 1
        scores = model.encode([q["query"]], normalize_embeddings=True) @ mat.T
        top = sorted(range(len(chunks)), key=lambda i: -scores[0][i])[:3]
        for rank, i in enumerate(top, 1):
            if owners[i] == q["doc"] and q["evidence"] in chunks[i]:
                hits += 1
                rr += 1 / rank
                break
    print("embedding sentence", split, "Hit@3:", round(hits / n, 3), "MRR@3:", round(rr / n, 3))

## Document chatbot demo

In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
chunks, ids, starts, ends = [], [], [], []
for doc_id, text in corpus.items():
    for i, (c, a, b) in enumerate(sentence_chunks(text)):
        chunks.append(c); ids.append(doc_id + ":" + str(i)); starts.append(a); ends.append(b)
vec = TfidfVectorizer(); mat = vec.fit_transform(chunks)
for q in ["How do I reset a forgotten password?", "What are tomorrow's flight prices?"]:
    scores = (vec.transform([q]) @ mat.T).toarray()[0]
    i = max(range(len(chunks)), key=lambda j: scores[j])
    print(q, ids[i], starts[i], ends[i], round(float(scores[i]), 3))
    print(chunks[i])
    if scores[i] < 0.2:
        print("No clear answer. Please contact the help desk.")
    else:
        print("Please confirm: is this right?")


## Scenarios

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer


class Retriever:
    def __init__(self, records, backend="tfidf"):
        if not records or any(not r["text"].strip() for r in records):
            raise ValueError("Provide nonempty records")
        self.records, self.backend = records, backend
        texts = [r["text"] for r in records]
        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(texts)
        elif backend == "embedding":
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
            self.check_length(texts)
            self.matrix = self.model.encode(texts, normalize_embeddings=True)
        else:
            raise ValueError("Use tfidf or embedding")

    def check_length(self, texts):
        for text in texts:
            n = len(self.model.tokenizer(text, truncation=False)["input_ids"])
            if n > self.model.max_seq_length:
                raise ValueError("Input exceeds encoder token limit")

    def search(self, query, k=3, filters=None):
        if not query.strip() or k <= 0:
            return []
        if self.backend == "tfidf":
            q = self.vectorizer.transform([query])
            if q.nnz == 0:
                return []
            scores = (q @ self.matrix.T).toarray()[0]
        else:
            self.check_length([query])
            q = self.model.encode([query], normalize_embeddings=True)[0]
            scores = self.matrix @ q
        allowed = [i for i, r in enumerate(self.records) if all(r.get(key) == value for key, value in (filters or {}).items())]
        ranked = sorted(allowed, key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in ranked[:k]]


def show_hits(hits):
    if not hits:
        print("No matching records.")
    for record, score in hits:
        print(record["id"], f"score={score:.3f}")
        print(record["text"])


In [ ]:

records = [{"id": "assessment:late", "source": "handbook.txt", "text": "Late submission is normally penalized. Students with illness may request an extension with medical evidence before the deadline."}]
engine = Retriever(records); show_hits(engine.search("illness medical evidence late submission", k=1))
records = [{"id": "printer:blank", "device": "printer", "text": "For blank printer pages: 1. Check ink level. 2. Run the printer cleaning utility. 3. Print a test page."}]
engine = Retriever(records); show_hits(engine.search("blank pages", k=1, filters={"device": "printer"}))
